[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/filip-borek/GNN_Projekt/blob/main/01_Eksperymenty.ipynb)

# Graph Neural Networks (MPNN) Experiments for LogP Prediction

This notebook documents the evaluation of various Graph Neural Network (GNN) architectures for predicting the partition coefficient (LogP) of chemical molecules. The aim is to identify the most effective model for this regression task by experimenting with different global pooling strategies and network depths.

**Objectives:**
- Load and preprocess the molecular dataset (converting SMILES to graph representations).
- Evaluate a baseline 3-layer MPNN with mean pooling.
- Test alternative global pooling operations (Summation and Multi-Pooling) to capture molecular size information.
- Analyze the impact of increasing network depth (5 layers vs 3 layers).
- Compare models using Validation Mean Squared Error (MSE) and Mean Absolute Error (MAE).

In [ ]:
import os
import torch

if not os.path.isdir("GNN_Projekt"):
    !git clone https://github.com/filip-borek/GNN_Projekt.git
if os.path.basename(os.getcwd()) != "GNN_Projekt":
    os.chdir("GNN_Projekt")

torch_version = torch.__version__.split("+")[0]
cuda = f"cu{torch.version.cuda.replace('.', '')}" if torch.cuda.is_available() else "cpu"
!pip install -q pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-{torch_version}+{cuda}.html
!pip install -q torch-geometric rdkit matplotlib tqdm requests

In [ ]:
import sys
import torch
from torch_geometric.loader import DataLoader
import matplotlib.pyplot as plt
from models.mpnn import MPNN, MPNNAdd, MPNNAdd5Layers, MPNNMultiPool

# Importing custom modules for data processing and model training
from data_processing import process_all_data
from trainer import train_model

In [ ]:
# Data preparation function call (abstracts away the data pipeline)
# The 'num_molecules' parameter controls the dataset size. Set to None to process the full dataset.
train_dataset, val_dataset, test_dataset = process_all_data(num_molecules=10000)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print(f"Train size: {len(train_dataset)}")
print(f"Val size:   {len(val_dataset)}")

## Experiment 1: Baseline MPNN (Mean Pooling)

**Architecture:** 3 Message Passing layers followed by Global Mean Pooling.
**Rationale:** Mean pooling averages the node features across the entire graph. It provides a solid baseline but might lose important extensive information (e.g., the total size of the molecule), as a small and a large molecule could theoretically yield similar mean node features.

In [ ]:
# Retrieve node and edge feature dimensions from the dataset
node_dim = train_dataset[0].x.shape[1]
edge_dim = train_dataset[0].edge_attr.shape[1]

print("Running Experiment 1: Baseline MPNN (Mean Pooling)...")
model_default = MPNN(node_dim=node_dim, edge_dim=edge_dim, hidden_dim=64, hidden_layer_dim=32)
wyniki_default = train_model(model_default, train_loader, val_loader, epochs=60)
train_losses, val_losses, train_accuracies, val_accuracies = wyniki_default

## Experiment 2: MPNN with Sum Pooling (Add Pooling)

**Architecture:** 3 Message Passing layers followed by Global Add (Sum) Pooling.
**Rationale:** LogP is an extensive property, meaning it scales with the size of the molecule. Sum pooling aggregates features by adding them up, which explicitly preserves the size information of the molecular graph and should theoretically perform better than mean pooling for this specific task.

In [ ]:
print("Running Experiment 2: MPNN with Sum Pooling...")
model_add = MPNNAdd(node_dim=node_dim, edge_dim=edge_dim, hidden_dim=64, hidden_layer_dim=32)
wyniki_add = train_model(model_add, train_loader, val_loader, epochs=60)
train_losses_add, val_losses_add, train_accuracies_add, val_accuracies_add = wyniki_add

## Experiment 3: Deeper MPNN (5 Layers, Sum Pooling)

**Architecture:** 5 Message Passing layers followed by Global Add Pooling.
**Rationale:** Increasing the depth of the network allows information to propagate further across the molecular graph (up to 5 hops). This might help capture more complex substructures. However, deeper GNNs can suffer from over-smoothing, so this experiment tests if the added depth is beneficial or detrimental.

In [ ]:
print("Running Experiment 3: 5-Layer MPNN with Sum Pooling...")
model_add_5l = MPNNAdd5Layers(node_dim=node_dim, edge_dim=edge_dim, hidden_dim=64, hidden_layer_dim=32)
wyniki_add_5l = train_model(model_add_5l, train_loader, val_loader, epochs=60)
train_losses_add_5l, val_losses_add_5l, train_accuracies_add_5l, val_accuracies_add_5l = wyniki_add_5l


## Experiment 4: MPNN with Multi-Pooling (Mean + Max + Add)

**Architecture:** 3 Message Passing layers followed by a concatenation of Mean, Max, and Add Pooling.
**Rationale:** Different pooling operations capture different aspects of the graph topology. By concatenating mean (average properties), max (most prominent features), and add (size information) pooling, the model receives a richer, more comprehensive representation of the entire graph.

In [ ]:
print("Running Experiment 4: MPNN with Multi-Pooling...")
model_multipool = MPNNMultiPool(node_dim=node_dim, edge_dim=edge_dim, hidden_dim=64, hidden_layer_dim=32)
wyniki_multipool = train_model(model_multipool, train_loader, val_loader, epochs=60)
train_losses_multi, val_losses_multi, train_accuracies_multi, val_accuracies_multi = wyniki_multipool


## Final Comparison and Conclusions

The plot below compares the Validation MSE and Validation MAE for all the tested architectures over 50 epochs.

**Observations:**
1. **Mean vs. Sum Pooling:** As hypothesized, Sum Pooling (Add) significantly outperforms Mean Pooling. LogP is an additive/extensive property, so retaining size information is crucial.
2. **Impact of Depth:** The 5-layer model does not necessarily outperform the 3-layer model and can be more unstable during training, likely due to the over-smoothing problem common in deeper GNNs.
3. **Multi-Pooling:** Combining all pooling strategies (Mean + Max + Add) yields very competitive results, often achieving the lowest loss, as it leverages the strengths of all aggregation methods.

In [ ]:
epochs_range = range(1, 61)

plt.figure(figsize=(16, 6))
plt.style.use('seaborn-v0_8-whitegrid')

# Plot Validation MSE
plt.subplot(1, 2, 1)
plt.plot(epochs_range, val_losses, label='Val MSE (3L Mean)', linestyle='-')
plt.plot(epochs_range, val_losses_add, label='Val MSE (3L Add)', linestyle='-')
plt.plot(epochs_range, val_losses_add_5l, label='Val MSE (5L Add)', linestyle='--')
plt.plot(epochs_range, val_losses_multi, label='Val MSE (3L Multi-Pool)', linestyle='-.', linewidth=2)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('MSE Loss', fontsize=12)
plt.title('Validation MSE Loss Comparison', fontsize=14)
plt.legend(fontsize=11)

# Plot Validation MAE
plt.subplot(1, 2, 2)
plt.plot(epochs_range, val_accuracies, label='Val MAE (3L Mean)', linestyle='-')
plt.plot(epochs_range, val_accuracies_add, label='Val MAE (3L Add)', linestyle='-')
plt.plot(epochs_range, val_accuracies_add_5l, label='Val MAE (5L Add)', linestyle='--')
plt.plot(epochs_range, val_accuracies_multi, label='Val MAE (3L Multi-Pool)', linestyle='-.', linewidth=2)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Mean Absolute Error (MAE)', fontsize=12)
plt.title('Validation MAE Comparison', fontsize=14)
plt.legend(fontsize=11)

plt.tight_layout()
plt.show()